<a href="https://colab.research.google.com/github/aslestia/Published_article/blob/main/ACS_Manuscript_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, time
RUN_ID = time.strftime("%Y%m%d_%H%M%S")
OUTDIR = f"/content/drive/MyDrive/rl_results/EVAL_{RUN_ID}"
os.makedirs(OUTDIR, exist_ok=True)
print("OUTDIR:", OUTDIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OUTDIR: /content/drive/MyDrive/rl_results/EVAL_20260215_145446


#CELL 1 — Import + seed util

In [ ]:
import numpy as np
import pandas as pd
import random
import gc

def set_seed(seed:int):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except Exception:
        pass


#CELL 2 — Ambil data & bentuk returns

In [ ]:
import yfinance as yf

tickers = ["AAPL","AMZN","GOOG","JPM","META","MSFT","NFLX","NVDA","TSLA","V"]
start, end = "2019-01-01", "2024-11-01"

prices = yf.download(tickers, start=start, end=end)["Close"].dropna()
returns = prices.pct_change().dropna()



/tmp/ipython-input-1832786353.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  prices = yf.download(tickers, start=start, end=end)["Close"].dropna()
[*********************100%***********************]  10 of 10 completed


#CELL 3 — Split train/eval (chronological)

In [ ]:
split_ratio = 0.8
T = len(returns)
cut = int(split_ratio*T)
returns_train = returns.iloc[:cut].copy()
returns_eval  = returns.iloc[cut:].copy()

print("Train:", returns_train.shape, "Eval:", returns_eval.shape)


Train: (1174, 10) Eval: (294, 10)


#CELL 4 — Environment (PENTING: tanpa registry gym/gymnasium)

In [ ]:
import gymnasium as gym
from gymnasium import spaces
from gymnasium.envs.registration import register
from sklearn.preprocessing import StandardScaler

class MultiStockReturnEnv(gym.Env):
    metadata = {'render.modes': ['human']}

    def __init__(self, returns_df, initial_balance=10000, transaction_cost_pct=0.001,
                 reward_scale=1e-3, max_share_unit=1.0, clip_ret=0.2, debug=False):
        super().__init__()
        # data
        self.raw_df = returns_df.reset_index(drop=True).copy()
        self.df = self.raw_df.fillna(0.0).astype(float)
        self.n_stocks = self.df.shape[1]

        # params
        self.initial_balance = float(initial_balance)
        self.transaction_cost_pct = float(transaction_cost_pct)
        self.reward_scale = float(reward_scale)
        self.max_share_unit = float(max_share_unit)  # scaling factor for holdings
        self.clip_ret = float(clip_ret)
        self.debug = bool(debug)

        # observation: normalized returns (n_stocks) + cash_norm + shares_norm_mean
        obs_dim = self.n_stocks + 2
        high = np.ones(obs_dim, dtype=np.float32) * np.finfo(np.float32).max
        self.observation_space = spaces.Box(low=-high, high=high, dtype=np.float32)

        # discrete action space (easy baseline)
        self.action_space = spaces.Discrete(3)
        self.scaler = StandardScaler()
        self.scaler.fit(self.df.values)

        # state
        self.reset()

    def _get_obs(self):
        row = np.nan_to_num(self.df.loc[self.current_step].values, nan=0.0).astype(np.float32)
        row = np.clip(row, -self.clip_ret, self.clip_ret)

        try:
            row_scaled = self.scaler.transform(row.reshape(1, -1)).flatten()
        except Exception:
            row_scaled = row

        cash_norm = np.clip(self.balance / (self.initial_balance * 10.0), -1e3, 1e3)  # bounded
        shares_norm = np.clip(np.mean(self.shares_held) / (self.max_share_unit + 1e-9), -1e3, 1e3)
        obs = np.concatenate([row_scaled.astype(np.float32), np.array([cash_norm, shares_norm], dtype=np.float32)])
        obs = np.nan_to_num(obs, nan=0.0, posinf=1e9, neginf=-1e9)
        return obs

    def step(self, action):
        info = {}

        rets = np.nan_to_num(self.df.loc[self.current_step].values, nan=0.0).astype(np.float64)
        rets = np.clip(rets, -self.clip_ret, self.clip_ret)

        prev_total = float(self.total_asset)

        if action == 1:  # buy all available cash, proportional across stocks
            cash_available = self.balance * (1.0 - self.transaction_cost_pct)
            units_per_stock = (cash_available / self.initial_balance) / max(1.0, self.n_stocks)
            self.shares_held += units_per_stock
            self.balance = 0.0
        elif action == 2:  # sell all (liquidate)
            mean_rets = np.mean(rets)
            proceeds = np.sum(self.shares_held) * self.initial_balance * (1.0 + mean_rets)
            proceeds *= (1.0 - self.transaction_cost_pct)
            self.balance += proceeds
            self.shares_held[:] = 0.0

        try:
            self.shares_held = self.shares_held * (1.0 + rets)
        except Exception:
            if np.isscalar(self.shares_held):
                self.shares_held = np.array([self.shares_held] * self.n_stocks)
                self.shares_held = self.shares_held * (1.0 + rets)
            else:
                self.shares_held = np.zeros(self.n_stocks)

        self.shares_held = np.nan_to_num(self.shares_held, nan=0.0, posinf=1e9, neginf=-1e9)
        self.total_asset = float(self.balance + np.sum(self.shares_held) * self.initial_balance)

        reward = (self.total_asset - prev_total) * self.reward_scale
        reward = float(np.nan_to_num(reward, nan=0.0))
        reward = float(np.clip(reward, -1.0, 1.0))

        if self.debug:
            if (np.isnan(reward) or np.isnan(self.total_asset) or np.isnan(self.shares_held).any()):
                print(f"[DEBUG NaN] step={self.current_step} reward={reward} total={self.total_asset}")
            if abs(reward) > 0.5:
                print(f"[DEBUG LargeReward] step={self.current_step} reward={reward} total={self.total_asset}")

        self.current_step += 1
        done = False
        if self.current_step >= len(self.df) - 1:
            done = True

        obs = self._get_obs()
        terminated, truncated = done, False
        return obs, reward, terminated, truncated, info

    def reset(self, seed=None, options=None):
        try:
            super().reset(seed=seed)
        except TypeError:
            pass
        self.current_step = 0
        self.balance = float(self.initial_balance)
        self.shares_held = np.zeros(self.n_stocks, dtype=np.float64)
        self.total_asset = float(self.initial_balance)
        obs = self._get_obs()
        return obs, {}

    def render(self, mode='human'):
        print(f"Step {self.current_step} | Total Asset: {self.total_asset:.2f}")

tickers = ['AAPL', 'MSFT', 'GOOG',
           'AMZN', 'META', 'TSLA',
           'NVDA', 'JPM', 'V', 'NFLX']

dfs = []
for t in tickers:
    df_t = yf.download(t, period='5y', interval='1d', progress=False)
    df_t = df_t[['Close']].rename(columns={'Close': t})
    dfs.append(df_t)

df_all = pd.concat(dfs, axis=1).dropna()
returns = df_all.pct_change().dropna().reset_index(drop=True)

env_id = "MultiStockReturn-v0"
if env_id in gym.registry:
    del gym.registry[env_id]

register(
    id=env_id,
    entry_point=lambda:
        MultiStockReturnEnv(
            returns_df=returns,
            reward_scale=1e-3
        ),
    max_episode_steps=len(returns) - 1,
)

print("Registered envs:", [spec.id for spec in gym.registry.values() if "MultiStockReturn" in spec.id])

env = gym.make(env_id)
obs, _ = env.reset()
print("obs shape:", obs.shape, "obs sample:", obs[:5])
step_out = env.step(0)
print("step output lengths:", len(step_out))

/tmp/ipython-input-13389218.py:125: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_t = yf.download(t, period='5y', interval='1d', progress=False)
/tmp/ipython-input-13389218.py:125: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_t = yf.download(t, period='5y', interval='1d', progress=False)
/tmp/ipython-input-13389218.py:125: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_t = yf.download(t, period='5y', interval='1d', progress=False)
/tmp/ipython-input-13389218.py:125: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_t = yf.download(t, period='5y', interval='1d', progress=False)
/tmp/ipython-input-13389218.py:125: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_t = yf.download(t, period='5y', interval='1d', progress=False)
/tmp/ipython-input-13389218.py:125: FutureWarning: YF.download() has changed argument auto

Registered envs: ['MultiStockReturn-v0']
obs shape: (12,) obs sample: [-1.048824    0.22981696  0.10284569  0.5287684  -0.09179907]
step output lengths: 5


In [ ]:
# quick sanity
import gymnasium as gym
env_test = MultiStockReturnEnv(returns_train, reward_scale=1e-3)
obs, _ = env_test.reset()
print("obs_dim:", obs.shape, "act_n:", env_test.action_space.n)


obs_dim: (12,) act_n: 3


In [ ]:
import torch
import torch.nn as nn

class RunningNorm:
    def __init__(self, eps=1e-5):
        self.mean = 0.0
        self.var = 1.0
        self.count = eps
    def update(self, x):
        x = np.array(x)
        batch_mean = x.mean()
        batch_var  = x.var()
        batch_count = 1
        delta = batch_mean - self.mean
        tot = self.count + batch_count
        new_mean = self.mean + delta * batch_count / tot
        m_a = self.var * self.count
        m_b = batch_var * batch_count
        M2 = m_a + m_b + delta**2 * self.count * batch_count / tot
        new_var = M2 / tot
        self.mean, self.var, self.count = new_mean, new_var, tot
    def norm(self, x):
        return (x - self.mean) / (np.sqrt(self.var) + 1e-8)

class Meter:
    def __init__(self, window=20):
        self.window = int(window)
        self.rewards = []
    def push(self, r):
        self.rewards.append(float(r))
    def get_last_moving_avg(self):
        if not self.rewards:
            return 0.0
        w = min(self.window, len(self.rewards))
        return float(np.mean(self.rewards[-w:]))

def make_optimizer(params, opt_name='adam', lr=1e-3, weight_decay=0.0):
    opt_name = opt_name.lower()
    if opt_name == 'adam':
        return torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)
    if opt_name == 'adamw':
        return torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    if opt_name == 'rmsprop':
        return torch.optim.RMSprop(params, lr=lr, weight_decay=weight_decay)
    if opt_name == 'sgd':
        return torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay)
    raise ValueError(f"Unknown optimizer: {opt_name}")

def reset_env(env, seed=None):
    out = env.reset(seed=seed)
    return out[0] if isinstance(out, tuple) else out

def step_env(env, action):
    out = env.step(action)
    # gymnasium: (obs, r, terminated, truncated, info)
    if len(out) == 5:
        obs, r, terminated, truncated, info = out
        done = terminated or truncated
        return obs, r, done, truncated, info
    # gym old: (obs, r, done, info)
    obs, r, done, info = out
    return obs, r, done, False, info

def linear_schedule(start, end, t, T):
    if T <= 0: return float(end)
    frac = min(max(t / T, 0.0), 1.0)
    return float(start + frac * (end - start))


#CELL 5 — Training functions (copy paste dari notebook lama)

##RunningNorm

In [ ]:
class RunningNorm:
    def __init__(self, eps=1e-5):
        self.mean = 0.0
        self.var = 1.0
        self.count = eps

    def update(self, x):
        x = np.array(x)
        batch_mean = x.mean()
        batch_var = x.var()
        batch_count = 1

        delta = batch_mean - self.mean
        tot_count = self.count + batch_count

        new_mean = self.mean + delta * batch_count / tot_count
        m_a = self.var * self.count
        m_b = batch_var * batch_count
        M2 = m_a + m_b + delta**2 * self.count * batch_count / tot_count
        new_var = M2 / tot_count

        self.mean = new_mean
        self.var = new_var
        self.count = tot_count

    def norm(self, x):
        return (x - self.mean) / (np.sqrt(self.var) + 1e-8)


##TAMBAHAN

In [ ]:
import numpy as np
import torch
import torch.nn as nn

def reset_env(env, seed=None):
    # gymnasium reset() -> (obs, info)
    out = env.reset(seed=seed)
    if isinstance(out, tuple) and len(out) == 2:
        return out[0]
    return out

def step_env(env, action):
    # gymnasium step() -> (obs, reward, terminated, truncated, info)
    out = env.step(action)
    if len(out) == 5:
        obs, r, terminated, truncated, info = out
        done = terminated or truncated
        return obs, r, done, truncated, info
    # fallback (gym lama)
    obs, r, done, info = out
    return obs, r, done, False, info

def linear_schedule(start, end, t, T):
    if T <= 0:
        return float(end)
    frac = min(max(t / T, 0.0), 1.0)
    return float(start + frac * (end - start))

class Meter:
    def __init__(self, window=20):
        self.window = int(window)
        self.rewards = []

    def push(self, r):
        self.rewards.append(float(r))

    def get_last_moving_avg(self):
        if len(self.rewards) == 0:
            return 0.0
        w = min(self.window, len(self.rewards))
        return float(np.mean(self.rewards[-w:]))

    def moving_avg(self):
        # optional helper
        out = []
        for i in range(len(self.rewards)):
            w = min(self.window, i+1)
            out.append(float(np.mean(self.rewards[i+1-w:i+1])))
        return out

def make_optimizer(params, opt_name='adam', lr=1e-3, weight_decay=0.0):
    opt_name = opt_name.lower()
    if opt_name == 'adam':
        return torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)
    if opt_name == 'adamw':
        return torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    if opt_name == 'rmsprop':
        return torch.optim.RMSprop(params, lr=lr, weight_decay=weight_decay)
    if opt_name == 'sgd':
        return torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay)
    raise ValueError(f"Unknown optimizer: {opt_name}")


##REINFORCE

In [ ]:
import torch
import torch.nn as nn
import math

class PolicyNet(nn.Module):
    """Jaringan Kebijakan murni (untuk REINFORCE tanpa baseline V-value)."""
    def __init__(self, obs_dim, act_dim, hidden=128, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.LayerNorm(hidden),
            nn.LeakyReLU(0.1),

            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
            nn.LeakyReLU(0.1),

            nn.Dropout(dropout),
            nn.Linear(hidden, act_dim)
        )

        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=1.0)
                nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        return torch.distributions.Categorical(logits=self.net(x))

class PolicyBaseline(nn.Module):
    """Jaringan Policy dan Value (untuk REINFORCE dengan baseline V-value)."""
    def __init__(self, obs_dim, act_dim, hidden=128):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(obs_dim, hidden), nn.ReLU())
        self.pi_head = nn.Linear(hidden, act_dim)
        self.v_head  = nn.Linear(hidden, 1)

    def forward(self, x):
        h = self.body(x)
        dist = torch.distributions.Categorical(logits=self.pi_head(h))
        v    = self.v_head(h).squeeze(-1)
        return dist, v

# --- Core RL Calculations ---

def compute_returns(rewards, gamma):
    """Menghitung discounted returns (G_t)."""
    G, out = 0.0, []
    for r in reversed(rewards):
        G = r + gamma * G
        out.append(G)
    out.reverse()
    return torch.tensor(out, dtype=torch.float32)

def compute_cvar_value(returns, alpha=0.1):
    """Menghitung CVaR dari returns (bukan loss)."""
    sorted_returns, _ = torch.sort(returns)
    n = len(sorted_returns)
    k = max(1, math.ceil(alpha * n))
    cvar = sorted_returns[:k].mean()
    return cvar


# --- Training Function ---
def train_reinforce(env, episodes=800, gamma=0.99, seed=42,
                                   opt_name='adam', lr=1e-3, weight_decay=0.0, hidden=256,
                                   use_baseline=True,
                                   var=False, cvar=False, evar=False,
                                   alpha=0.1, beta_risk=0.5, # alpha untuk VaR/CVaR, beta_risk untuk EVaR
                                   entropy_start=0.02, entropy_end=0.001,
                                   grad_clip=0.5, render=False, window=20,
                                   csv_filename='reinforce_risk_log.csv'):

    # -----------------------------------------------------------
    # PENGECEKAN KENDALA RISIKO (CONSTRAINTS)
    # -----------------------------------------------------------
    risk_options = [var, cvar, evar]
    active_risks = sum(risk_options)

    if active_risks > 1:
        print("-------------------------------------------------------------------------------------------")
        print("ERROR: Hanya boleh memilih SATU ukuran risiko (VAR, CVAR, atau EVAR) aktif pada satu waktu.")
        print("-------------------------------------------------------------------------------------------")
        return [], [], None

    if var: risk_measure = 'var'
    elif cvar: risk_measure = 'cvar'
    elif evar: risk_measure = 'evar'
    else: risk_measure = None

    if use_baseline:
        tag = 'REINFORCE-BL'
        if risk_measure: tag += f"-{risk_measure.upper()}"
    else:
        tag = 'REINFORCE'
        if risk_measure: tag += f"-{risk_measure.upper()}"

    # The previous diff introduced an indentation error and used an undefined env_name. Reverting to safe check.
    env_name_str = env.spec.id if hasattr(env, 'spec') and env.spec else "CustomEnv" # Safely get env name
    print(f"Memulai pelatihan {tag} di {env_name_str}.")


    # --- Setup Awal ---
    set_seed(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Menggunakan perangkat: {device}")

    #env = gym.make(env_name)
    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.n
    stnorm = RunningNorm()

    if use_baseline:
        net = PolicyBaseline(obs_dim, act_dim, hidden).to(device)
    else:
        net = PolicyNet(obs_dim, act_dim, hidden).to(device)

    params = net.parameters()
    opt = make_optimizer(params, opt_name=opt_name, lr=lr, weight_decay=weight_decay)
    meter = Meter(window=window)

    log_data = []

    # --- Training Loop ---
    for ep in range(episodes):
        obs = reset_env(env, seed+ep)
        done, total_r = False, 0.0

        logps, rewards, entros, values = [], [], [], []

        while not done:
            stnorm.update(obs)
            obs_t = torch.tensor(stnorm.norm(obs), dtype=torch.float32, device=device).unsqueeze(0)

            if use_baseline:
                dist, v = net(obs_t)
                values.append(v.squeeze(0))
            else:
                dist = net(obs_t)

            a = dist.sample()

            logps.append(dist.log_prob(a).squeeze(0))
            entros.append(dist.entropy().squeeze(0))

            nobs, r, done, truncated, info = step_env(env, a.item())
            done = done or truncated
            rewards.append(float(r))
            total_r += r
            obs = nobs

        # --- Perhitungan Loss ---

        # 1. Hitung Discounted Returns (G)
        returns = compute_returns(rewards, gamma).to(device)

        # 2. Hitung Advantage (Advantage) atau Normalized Return
        if use_baseline:
            values_t = torch.stack(values)
            adv = returns - values_t.detach() # Baseline G - V
        else:
            # Default: Risk Neutral REINFORCE
            adv = (returns - returns.mean()) / (returns.std() + 1e-8)

        # 3. PENYESUAIAN RISIKO (RISK-SENSITIVE POLICY GRADIENT)
        if risk_measure:

            # Tentukan Baseline Risiko:
            if risk_measure == 'cvar':
                risk_baseline = compute_cvar_value(returns, alpha)
            elif risk_measure == 'var':
                risk_baseline = torch.quantile(returns, alpha)
            elif risk_measure == 'evar':
                # Untuk EVaR, kita menggunakan mean return sebagai baseline untuk perhitungan adv
                # tapi penyesuaian utama ada di weighting (Langkah 3b)
                risk_baseline = returns.mean()

            # 3a. Re-baseline Advantage menggunakan Risk Baseline (Jika tanpa V-value)
            if not use_baseline and risk_measure in ['cvar', 'var', 'evar']:
                # Hentikan normalisasi Mean/Std standar, dan gunakan Risk Baseline
                adv = returns - risk_baseline
                # Normalisasi adv hasil risk-baseline untuk stabilitas (penting!)
                adv = (adv - adv.mean()) / (adv.std() + 1e-8)


            # 3b. Weighting / Masking
            if risk_measure == 'cvar':
                # CVaR: HANYA fokus pada pembaruan dari returns yang berada di kuantil terburuk (alpha) (bagian kiri, kuantil bawah)
                sorted_returns, _ = torch.sort(returns, descending=False)
                k = max(1, math.ceil(alpha * len(returns)))
                cvar_threshold = sorted_returns[k-1]

                # Masking: Advantage hanya aktif (1.0) untuk langkah yang Returns-nya di bawah threshold
                mask = returns <= cvar_threshold
                adv = adv * mask.float()

            elif risk_measure == 'var':
                # VaR: HANYA fokus pada pembaruan dari returns yang lebih buruk dari VaR (kuantil alpha)
                var_threshold = torch.quantile(returns, alpha)

                # Masking: Advantage hanya aktif (1.0) untuk langkah yang Returns-nya di bawah VaR
                mask = returns <= var_threshold
                adv = adv * mask.float()

            elif risk_measure == 'evar':
                adv = adv * torch.exp(beta_risk * adv.detach()) # Exponential weighting


        # Hitung Policy Loss
        policy_loss = -(torch.stack(logps) * adv.detach()).sum()

        # Hitung Value Loss (Hanya jika menggunakan PolicyBaseline)
        if use_baseline:
            value_loss = 0.5 * (adv.pow(2)).sum()
            loss_core = policy_loss + value_loss
        else:
            value_loss = torch.tensor(0.0, device=device)
            loss_core = policy_loss

        # Hitung Entropy Loss
        ent = torch.stack(entros).sum()
        warmup = int(0.7 * episodes)
        if ep <= warmup:
            beta = linear_schedule(entropy_start, entropy_end, ep, warmup)
        else:
            beta = entropy_end

        loss = loss_core - beta * ent

        # Backpropagation
        opt.zero_grad()
        loss.backward()
        if grad_clip is not None:
            nn.utils.clip_grad_norm_(params, grad_clip)
        opt.step()

        # Logging dan Monitoring
        meter.push(total_r)
        current_avg = meter.get_last_moving_avg()

        # Kumpulkan data log
        log_data.append({
            'episode': ep + 1,
            'reward': total_r,
            'moving_avg': current_avg,
            'policy_loss': policy_loss.item(),
            'value_loss': value_loss.item(),
            'total_loss': loss.item(),
            'entropy_beta': beta,
            'risk_measure': risk_measure if risk_measure else 'None',
        })

        if (ep+1) % 25 == 0:
            risk_info = f"| Risk: {risk_measure.upper()}" if risk_measure else ""
            print(f'[{tag}] Ep {ep+1}/{episodes} | R:{total_r:.1f} | Avg({window}): {current_avg:.1f} | Ent:{beta:.4f} {risk_info}')

            # Make CartPole-v1 check safe for environments without spec
            if hasattr(env, 'spec') and env.spec and env.spec.id == 'CartPole-v1' and len(meter.rewards) >= 100 and np.mean(meter.rewards[-100:]) >= 475:
                print(f"[{tag}] Early stop: moving-avg(100) >= 475")
                break


    env.close()
    if csv_filename and log_data:
        fieldnames = log_data[0].keys()
        try:
            import csv # Import csv here if it's not global
            with open(csv_filename, 'w', newline='') as csvfile:
                writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
                writer.writeheader()
                writer.writerows(log_data)
            print(f"\n[{tag}] Pelatihan Selesai. Data log disimpan ke: {csv_filename}")
        except Exception as e:
            print(f"\n[{tag}] ERROR saat menyimpan CSV: {e}")

    # Return a DataFrame directly
    return pd.DataFrame(log_data), net


##A2C

In [ ]:
import torch
import torch.nn as nn
import math
import pandas as pd # Add pandas import for DataFrame logging
import csv # Add csv import for logging

class ActorCriticNet(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=256):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(obs_dim, hidden), nn.ReLU(),
                                  nn.Linear(hidden, hidden), nn.ReLU())
        self.pi_head = nn.Linear(hidden, act_dim)
        self.v_head  = nn.Linear(hidden, 1)
    def forward(self, x):
        h = self.body(x)
        dist = torch.distributions.Categorical(logits=self.pi_head(h))
        v    = self.v_head(h).squeeze(-1)
        return dist, v

def train_actor_critic_batched(env, episodes=800, gamma=0.99, seed=7,
                               opt_name='adam', lr=1e-3, weight_decay=0.0, hidden=256,
                               value_loss_coeff=0.7, entropy_start=0.02, entropy_end=0.005,
                               grad_clip=0.5, render=False, window=20,
                               var=False, cvar=False, evar=False, # New risk parameters
                               alpha=0.1, beta_risk=0.5,           # New risk parameters
                               csv_filename='a2c_risk_log.csv'): # New CSV logging parameter

    # -----------------------------------------------------------
    # PENGECEKAN KENDALA RISIKO (CONSTRAINTS)
    # -----------------------------------------------------------
    risk_options = [var, cvar, evar]
    active_risks = sum(risk_options)

    if active_risks > 1:
        print("-------------------------------------------------------------------------------------------")
        print("ERROR: Hanya boleh memilih SATU ukuran risiko (VAR, CVAR, atau EVAR) aktif pada satu waktu.")
        print("-------------------------------------------------------------------------------------------")
        # Returning an empty DataFrame to match the expected return type
        return pd.DataFrame(), None # Return None for net as well to match signature

    if var: risk_measure = 'var'
    elif cvar: risk_measure = 'cvar'
    elif evar: risk_measure = 'evar'
    else: risk_measure = None

    tag = 'A2C'
    if risk_measure: tag += f"-{risk_measure.upper()}"

    env_name_str = env.spec.id if hasattr(env, 'spec') and env.spec else "CustomEnv"
    print(f"Memulai pelatihan {tag} di {env_name_str}.")


    set_seed(seed)
    device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Menggunakan perangkat: {device}")

    obs_dim  = env.observation_space.shape[0]
    act_dim  = env.action_space.n
    stnorm   = RunningNorm()

    net   = ActorCriticNet(obs_dim, act_dim, hidden).to(device)
    opt   = make_optimizer(net.parameters(), opt_name=opt_name, lr=lr, weight_decay=weight_decay)
    meter = Meter(window=window)

    log_data = [] # Initialize log_data

    for ep in range(episodes):
        obs = reset_env(env, seed+ep)
        done, total_r = False, 0.0
        logps, values, rewards, entros = [], [], [], []

        while not done:
            stnorm.update(obs)
            obs_t = torch.tensor(stnorm.norm(obs), dtype=torch.float32, device=device).unsqueeze(0)
            dist, v = net(obs_t)
            a = dist.sample()
            logps.append(dist.log_prob(a).squeeze(0))
            entros.append(dist.entropy().squeeze(0))
            values.append(v.squeeze(0))

            nobs, r, done_step, truncated, info = step_env(env, a.item()) # Get all 5 step outputs
            done = done_step or truncated # Update done based on both terminated and truncated
            rewards.append(float(r))
            total_r += r

            obs = nobs # Move observation forward

        # --- AFTER EPISODE ENDS ---
        # 1. Compute discounted returns for the episode (G_t)
        returns = compute_returns(rewards, gamma).to(device)

        # 2. Calculate targets for the value network: G_t
        targets = returns

        # 3. Calculate advantages (deltas): G_t - V(s_t)
        values_t = torch.stack(values)
        deltas = targets - values_t # This is our advantage, A_t

        # 4. PENYESUAIAN RISIKO (RISK-SENSITIVE POLICY GRADIENT)
        if risk_measure:
            # 4b. Weighting / Masking
            if risk_measure == 'cvar':
                # CVaR: HANYA fokus pada pembaruan dari returns yang berada di kuantil terburuk (alpha)
                sorted_returns, _ = torch.sort(returns, descending=False)
                k = max(1, math.ceil(alpha * len(returns)))
                cvar_threshold = sorted_returns[k-1]

                # Masking: Advantage only active for steps where Returns are below the threshold
                mask = returns <= cvar_threshold
                deltas = deltas * mask.float()

            elif risk_measure == 'var':
                # VaR: HANYA fokus pada pembaruan dari returns yang lebih buruk dari VaR (kuantil alpha)
                var_threshold = torch.quantile(returns, alpha)

                # Masking: Advantage only active for steps where Returns are below VaR
                mask = returns <= var_threshold
                deltas = deltas * mask.float()

            elif risk_measure == 'evar':
                # For EVaR, we apply exponential weighting to the advantages.
                deltas = deltas * torch.exp(beta_risk * deltas.detach())

        beta = linear_schedule(entropy_start, entropy_end, ep, episodes-1)
        actor_loss  = -(torch.stack(logps) * deltas.detach()).sum() # deltas already has risk adjustment
        critic_loss = (deltas.pow(2)).sum() * value_loss_coeff # Value loss is still based on squared advantage
        entropy_loss = -beta * torch.stack(entros).sum()
        loss = actor_loss + critic_loss + entropy_loss

        opt.zero_grad(); loss.backward()
        if grad_clip is not None:
            nn.utils.clip_grad_norm_(net.parameters(), grad_clip)
        opt.step()

        meter.push(total_r)
        current_avg = meter.get_last_moving_avg()

        # Kumpulkan data log
        log_data.append({
            'episode': ep + 1,
            'reward': total_r,
            'moving_avg': current_avg,
            'actor_loss': actor_loss.item(),
            'critic_loss': critic_loss.item(),
            'total_loss': loss.item(),
            'entropy_beta': beta,
            'risk_measure': risk_measure if risk_measure else 'None',
        })

        if (ep+1) % 50 == 0:
            risk_info = f"| Risk: {risk_measure.upper()}" if risk_measure else ""
            print(f'[{tag}] Ep {ep+1}/{episodes} | R:{total_r:.1f} | Avg({meter.window}): {current_avg:.1f} | Ent:{beta:.4f} {risk_info}')

    env.close()

    if csv_filename and log_data:
        fieldnames = log_data[0].keys()
        try:
            with open(csv_filename, 'w', newline='') as csvfile:
                writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
                writer.writeheader()
                writer.writerows(log_data)
            print(f"\n[{tag}] Pelatihan Selesai. Data log disimpan ke: {csv_filename}")
        except Exception as e:
            print(f"\n[{tag}] ERROR saat menyimpan CSV: {e}")

    # Return a DataFrame directly
    # The previous code was attempting to access 'meter.rewards' outside its scope.
    # Instead, we should return the 'log_data' which is already collected.
    log_df = pd.DataFrame(log_data)
    return log_df, net


#CELL 6 — Runner multi-seed (langsung simpan per seed ke Drive)

In [ ]:
def run_one(seed, algo, obj, episodes=400):
    set_seed(seed)

    env_tr = MultiStockReturnEnv(returns_train, reward_scale=1e-3)

    var_flag  = (obj == "var")
    cvar_flag = (obj == "cvar")
    evar_flag = (obj == "evar")

    csv_path   = os.path.join(OUTDIR, f"log_{algo}_{obj}_seed{seed}.csv")
    model_path = os.path.join(OUTDIR, f"model_{algo}_{obj}_seed{seed}.pt")

    if algo == "reinforce":
        df_log, net = train_reinforce(
            env=env_tr,
            episodes=episodes,
            seed=seed,
            use_baseline=True,
            var=var_flag, cvar=cvar_flag, evar=evar_flag,
            csv_filename=csv_path
        )

    elif algo == "a2c":
        # NOTE: kalau fungsi A2C Anda belum support var/cvar/evar/csv_filename,
        # hapus argumen tersebut.
        df_log, net = train_actor_critic_batched(
            env=env_tr,
            episodes=episodes,
            seed=seed,
            var=var_flag, cvar=cvar_flag, evar=evar_flag,
            csv_filename=csv_path
        )
    else:
        raise ValueError("algo harus 'reinforce' atau 'a2c'")

    # pastikan log tersimpan
    if isinstance(df_log, pd.DataFrame):
        df_log.to_csv(csv_path, index=False)

    # simpan model
    torch.save({
        "algo": algo,
        "obj": obj,
        "seed": seed,
        "model_state": net.state_dict(),
    }, model_path)

    last20 = float(pd.Series(df_log["reward"]).tail(20).mean()) if "reward" in df_log.columns else np.nan

    return {
        "seed": seed, "algo": algo, "obj": obj,
        "final_reward_last20": last20,
        "csv_log": csv_path,
        "model_path": model_path
    }


#CELL 7 - Jalankan Training

In [ ]:
SEEDS = [0,1,2]                # cukup dulu (nanti bisa tambah 3,4)
ALGOS = ["reinforce","a2c"]
OBJS  = ["rn","cvar"]
EPISODES = 400                 # 300–500 aman

rows = []
for algo in ALGOS:
    for obj in OBJS:
        for seed in SEEDS:
            print(f"Running algo={algo} obj={obj} seed={seed}")
            out = run_one(seed, algo, obj, episodes=EPISODES)
            rows.append(out)
            gc.collect()

df_models = pd.DataFrame(rows)
df_models.to_csv(os.path.join(OUTDIR, "models_and_logs_index.csv"), index=False)
df_models


Running algo=reinforce obj=rn seed=0
Memulai pelatihan REINFORCE-BL di CustomEnv.
Menggunakan perangkat: cpu


/tmp/ipython-input-621029446.py:234: UserWarning: `parameters` is an empty generator, no gradient clipping will occur.
  nn.utils.clip_grad_norm_(params, grad_clip)
/tmp/ipython-input-621029446.py:234: UserWarning: `parameters` is an empty generator, no gradient clipping will occur.
  nn.utils.clip_grad_norm_(params, grad_clip)
/tmp/ipython-input-621029446.py:234: UserWarning: `parameters` is an empty generator, no gradient clipping will occur.
  nn.utils.clip_grad_norm_(params, grad_clip)
/tmp/ipython-input-621029446.py:234: UserWarning: `parameters` is an empty generator, no gradient clipping will occur.
  nn.utils.clip_grad_norm_(params, grad_clip)
/tmp/ipython-input-621029446.py:234: UserWarning: `parameters` is an empty generator, no gradient clipping will occur.
  nn.utils.clip_grad_norm_(params, grad_clip)
/tmp/ipython-input-621029446.py:234: UserWarning: `parameters` is an empty generator, no gradient clipping will occur.
  nn.utils.clip_grad_norm_(params, grad_clip)
/tmp/ipyth

[REINFORCE-BL] Ep 25/400 | R:68.6 | Avg(20): 49.6 | Ent:0.0184 


/tmp/ipython-input-621029446.py:234: UserWarning: `parameters` is an empty generator, no gradient clipping will occur.
  nn.utils.clip_grad_norm_(params, grad_clip)
/tmp/ipython-input-621029446.py:234: UserWarning: `parameters` is an empty generator, no gradient clipping will occur.
  nn.utils.clip_grad_norm_(params, grad_clip)
/tmp/ipython-input-621029446.py:234: UserWarning: `parameters` is an empty generator, no gradient clipping will occur.
  nn.utils.clip_grad_norm_(params, grad_clip)
/tmp/ipython-input-621029446.py:234: UserWarning: `parameters` is an empty generator, no gradient clipping will occur.
  nn.utils.clip_grad_norm_(params, grad_clip)
/tmp/ipython-input-621029446.py:234: UserWarning: `parameters` is an empty generator, no gradient clipping will occur.
  nn.utils.clip_grad_norm_(params, grad_clip)
/tmp/ipython-input-621029446.py:234: UserWarning: `parameters` is an empty generator, no gradient clipping will occur.
  nn.utils.clip_grad_norm_(params, grad_clip)
/tmp/ipyth

[REINFORCE-BL] Ep 50/400 | R:123.7 | Avg(20): 98.9 | Ent:0.0167 


...

/tmp/ipython-input-621029446.py:234: UserWarning: `parameters` is an empty generator, no gradient clipping will occur.
  nn.utils.clip_grad_norm_(params, grad_clip)
/tmp/ipython-input-621029446.py:234: UserWarning: `parameters` is an empty generator, no gradient clipping will occur.
  nn.utils.clip_grad_norm_(params, grad_clip)
/tmp/ipython-input-621029446.py:234: UserWarning: `parameters` is an empty generator, no gradient clipping will occur.
  nn.utils.clip_grad_norm_(params, grad_clip)
/tmp/ipython-input-621029446.py:234: UserWarning: `parameters` is an empty generator, no gradient clipping will occur.
  nn.utils.clip_grad_norm_(params, grad_clip)
/tmp/ipython-input-621029446.py:234: UserWarning: `parameters` is an empty generator, no gradient clipping will occur.
  nn.utils.clip_grad_norm_(params, grad_clip)
/tmp/ipython-input-621029446.py:234: UserWarning: `parameters` is an empty generator, no gradient clipping will occur.
  nn.utils.clip_grad_norm_(params, grad_clip)
/tmp/ipyth

[REINFORCE-BL-CVAR] Ep 375/400 | R:39.8 | Avg(20): 46.3 | Ent:0.0010 | Risk: CVAR


/tmp/ipython-input-621029446.py:234: UserWarning: `parameters` is an empty generator, no gradient clipping will occur.
  nn.utils.clip_grad_norm_(params, grad_clip)
/tmp/ipython-input-621029446.py:234: UserWarning: `parameters` is an empty generator, no gradient clipping will occur.
  nn.utils.clip_grad_norm_(params, grad_clip)
/tmp/ipython-input-621029446.py:234: UserWarning: `parameters` is an empty generator, no gradient clipping will occur.
  nn.utils.clip_grad_norm_(params, grad_clip)
/tmp/ipython-input-621029446.py:234: UserWarning: `parameters` is an empty generator, no gradient clipping will occur.
  nn.utils.clip_grad_norm_(params, grad_clip)
/tmp/ipython-input-621029446.py:234: UserWarning: `parameters` is an empty generator, no gradient clipping will occur.
  nn.utils.clip_grad_norm_(params, grad_clip)
/tmp/ipython-input-621029446.py:234: UserWarning: `parameters` is an empty generator, no gradient clipping will occur.
  nn.utils.clip_grad_norm_(params, grad_clip)
/tmp/ipyth

[REINFORCE-BL-CVAR] Ep 400/400 | R:44.6 | Avg(20): 48.4 | Ent:0.0010 | Risk: CVAR

[REINFORCE-BL-CVAR] Pelatihan Selesai. Data log disimpan ke: /content/drive/MyDrive/rl_results/EVAL_20260215_100113/log_reinforce_cvar_seed2.csv
Running algo=a2c obj=rn seed=0
Memulai pelatihan A2C di CustomEnv.
Menggunakan perangkat: cpu
[A2C] Ep 50/400 | R:68.7 | Avg(20): 56.6 | Ent:0.0182 
[A2C] Ep 100/400 | R:124.7 | Avg(20): 113.4 | Ent:0.0163 
[A2C] Ep 150/400 | R:170.9 | Avg(20): 164.4 | Ent:0.0144 
[A2C] Ep 200/400 | R:173.3 | Avg(20): 172.1 | Ent:0.0125 
[A2C] Ep 250/400 | R:205.0 | Avg(20): 174.9 | Ent:0.0106 
[A2C] Ep 300/400 | R:194.6 | Avg(20): 177.4 | Ent:0.0088 
[A2C] Ep 350/400 | R:194.1 | Avg(20): 191.1 | Ent:0.0069 
[A2C] Ep 400/400 | R:200.6 | Avg(20): 196.6 | Ent:0.0050 

[A2C] Pelatihan Selesai. Data log disimpan ke: /content/drive/MyDrive/rl_results/EVAL_20260215_100113/log_a2c_rn_seed0.csv
Running algo=a2c obj=rn seed=1
Memulai pelatihan A2C di CustomEnv.
Menggunakan perangkat: cpu

,seed,algo,obj,final_reward_last20,csv_log,model_path
0,0,reinforce,rn,192.637432,/content/drive/MyDrive/rl_results/EVAL_2026021...,/content/drive/MyDrive/rl_results/EVAL_2026021...
1,1,reinforce,rn,191.678227,/content/drive/MyDrive/rl_results/EVAL_2026021...,/content/drive/MyDrive/rl_results/EVAL_2026021...
2,2,reinforce,rn,199.362310,/content/drive/MyDrive/rl_results/EVAL_2026021...,/content/drive/MyDrive/rl_results/EVAL_2026021...
3,0,reinforce,cvar,9.037552,/content/drive/MyDrive/rl_results/EVAL_2026021...,/content/drive/MyDrive/rl_results/EVAL_2026021...
4,1,reinforce,cvar,6.273652,/content/drive/MyDrive/rl_results/EVAL_2026021...,/content/drive/MyDrive/rl_results/EVAL_2026021...
5,2,reinforce,cvar,48.426127,/content/drive/MyDrive/rl_results/EVAL_2026021...,/content/drive/MyDrive/rl_results/EVAL_2026021...
6,0,a2c,rn,196.557228,/content/drive/MyDrive/rl_results/EVAL_2026021...,/content/drive/MyDrive/rl_results/EVAL_2026021...
7,1,a2c,rn,193.509539,/content/drive/MyDrive/rl_results/EVAL_2026021...,/content/drive/MyDrive/rl_results/EVAL_2026021...
8,2,a2c,rn,183.686348,/content/drive/MyDrive/rl_results/EVAL_2026021...,/content/drive/MyDrive/rl_results/EVAL_2026021...
9,0,a2c,cvar,18.671226,/content/drive/MyDrive/rl_results/EVAL_2026021...,/content/drive/MyDrive/rl_results/EVAL_2026021...


#CELL 8 - Metrik Portofolio dan Evaluator

In [ ]:
import math

def max_drawdown(equity):
    equity = np.asarray(equity, dtype=float)
    peak = np.maximum.accumulate(equity)
    dd = (equity - peak) / peak
    return float(dd.min()), dd

def cagr(equity, periods_per_year=252):
    equity = np.asarray(equity, dtype=float)
    n = len(equity) - 1
    if n <= 0: return np.nan
    return float((equity[-1]/equity[0])**(periods_per_year/n) - 1.0)

def ann_vol(daily_ret, periods_per_year=252):
    r = np.asarray(daily_ret, dtype=float)
    return float(np.std(r, ddof=1) * np.sqrt(periods_per_year)) if len(r) > 1 else np.nan

def sharpe(daily_ret, rf=0.0, periods_per_year=252):
    r = np.asarray(daily_ret, dtype=float) - rf/periods_per_year
    if len(r) <= 1 or np.std(r, ddof=1) == 0: return np.nan
    return float(np.mean(r)/np.std(r, ddof=1) * np.sqrt(periods_per_year))

def realized_var_cvar(daily_ret, alpha=0.05):
    r = np.asarray(daily_ret, dtype=float)
    L = -r
    var = np.quantile(L, 1 - alpha)
    cvar = L[L >= var].mean() if np.any(L >= var) else np.nan
    return float(var), float(cvar)

def realized_evar(daily_ret, alpha=0.05, t_grid=None):
    r = np.asarray(daily_ret, dtype=float)
    L = -r
    if t_grid is None:
        t_grid = np.logspace(-3, 2, 200)
    vals = []
    for t in t_grid:
        m = np.mean(np.exp(t * L))
        if not np.isfinite(m) or m <= 0:
            continue
        vals.append((1.0/t) * (np.log(m) - np.log(alpha)))
    return float(np.min(vals)) if vals else np.nan

def greedy_action_from_net(net, obs_t):
    out = net(obs_t)
    dist = out[0] if isinstance(out, tuple) else out
    probs = dist.probs.detach().cpu().numpy().reshape(-1)
    return int(np.argmax(probs))

def eval_policy(net, env_eval, device="cpu"):
    net.eval()
    obs, _ = env_eval.reset()
    equity = [float(env_eval.total_asset)]
    daily_ret = []
    done = False
    while not done:
        obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        a = greedy_action_from_net(net, obs_t)
        nobs, r, terminated, truncated, info = env_eval.step(a)
        done = terminated or truncated
        equity.append(float(env_eval.total_asset))
        if len(equity) >= 2:
            daily_ret.append(equity[-1]/equity[-2] - 1.0)
        obs = nobs

    mdd, dd_series = max_drawdown(equity)
    VaR, CVaR = realized_var_cvar(daily_ret, alpha=0.05)
    EVaR = realized_evar(daily_ret, alpha=0.05)
    return {
        "final_equity": equity[-1],
        "CAGR": cagr(equity),
        "Vol": ann_vol(daily_ret),
        "Sharpe": sharpe(daily_ret),
        "MDD": mdd,
        "VaR_5": VaR,
        "CVaR_5": CVaR,
        "EVaR_5": EVaR,
        "equity": equity,
        "drawdown": dd_series.tolist(),
        "daily_ret": daily_ret
    }


#CELL 9 — Load model .pt dan evaluasi di returns_eval

In [ ]:
import glob
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
env_eval = MultiStockReturnEnv(returns_eval, reward_scale=1e-3)

# ambil semua model yang barusan disimpan
model_files = sorted(glob.glob(os.path.join(OUTDIR, "model_*_seed*.pt")))
print("Found model files:", len(model_files))

results = []
for mp in model_files:
    ckpt = torch.load(mp, map_location=device)
    algo, obj, seed = ckpt["algo"], ckpt["obj"], ckpt["seed"]

    obs_dim = env_eval.observation_space.shape[0]
    act_dim = env_eval.action_space.n

    if algo == "reinforce":
        # Correct the hidden dimension to match training (256)
        net = PolicyBaseline(obs_dim, act_dim, hidden=256).to(device)
    elif algo == "a2c":
        # Correct the hidden dimension to match training (256)
        net = ActorCriticNet(obs_dim, act_dim, hidden=256).to(device)
    else:
        raise ValueError(algo)

    net.load_state_dict(ckpt["model_state"])

    out = eval_policy(net, env_eval, device=device)

    row = {k: out[k] for k in ["final_equity","CAGR","Vol","Sharpe","MDD","VaR_5","CVaR_5","EVaR_5"]}
    row.update({"algo": algo, "obj": obj, "seed": seed, "model_path": mp})
    results.append(row)

    # simpan plot equity & drawdown
    plt.figure()
    plt.plot(out["equity"])
    plt.title(f"Equity | {algo}-{obj} | seed={seed}")
    plt.xlabel("t"); plt.ylabel("equity")
    plt.savefig(os.path.join(OUTDIR, f"equity_{algo}_{obj}_seed{seed}.png"), dpi=160)
    plt.close()

    plt.figure()
    plt.plot(out["drawdown"])
    plt.title(f"Drawdown | {algo}-{obj} | seed={seed}")
    plt.xlabel("t"); plt.ylabel("drawdown")
    plt.savefig(os.path.join(OUTDIR, f"drawdown_{algo}_{obj}_seed{seed}.png"), dpi=160)
    plt.close()

df_eval = pd.DataFrame(results)
df_eval.to_csv(os.path.join(OUTDIR, "eval_metrics_all.csv"), index=False)
df_eval

Found model files: 0


""


#CELL 10 — Tabel mean ± std (ini masuk jurnal)

In [ ]:
summary = (df_eval
           .groupby(["algo","obj"])[["CAGR","Vol","Sharpe","MDD","VaR_5","CVaR_5","EVaR_5","final_equity"]]
           .agg(["mean","std"])
           .reset_index())
summary


KeyError: 'algo'

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(f"{OUTDIR}/eval_metrics_all.csv")

# 1) cek run degenerate (Vol=0)
df["degenerate"] = (df["Vol"] == 0) | (df["Vol"].isna())
print("Degenerate runs per group:")
print(df.groupby(["algo","obj"])["degenerate"].mean().rename("degenerate_rate"))

# 2) ringkasan mean±std + median + IQR (lebih reviewer-proof)
def q25(x): return np.nanquantile(x, 0.25)
def q50(x): return np.nanquantile(x, 0.50)
def q75(x): return np.nanquantile(x, 0.75)

metrics = ["CAGR","Vol","Sharpe","MDD","VaR_5","CVaR_5","EVaR_5","final_equity"]

summary = (df.groupby(["algo","obj"])[metrics]
             .agg(["mean","std", q25, q50, q75])
             .reset_index())

summary.to_csv(f"{OUTDIR}/portfolio_metrics_summary.csv", index=False)
summary


In [ ]:
df_nd = df[~df["degenerate"]].copy()

summary_nd = (df_nd.groupby(["algo","obj"])[metrics]
                .agg(["mean","std"])
                .reset_index())

summary_nd.to_csv(f"{OUTDIR}/portfolio_metrics_summary_no_degenerate.csv", index=False)
summary_nd


In [ ]:
df_nd = df[~df["degenerate"]].copy()

(df_nd.groupby(["algo","obj"])
     .agg(n=("seed","count"),
          seeds=("seed", lambda s: sorted(s.tolist())))
     .reset_index())


,algo,obj,n,seeds
0,a2c,cvar,2,"[1, 2]"
1,a2c,rn,3,"[0, 1, 2]"
2,reinforce,cvar,1,[2]
3,reinforce,rn,3,"[0, 1, 2]"


In [ ]:
df["degenerate"] = (df["Vol"] == 0) | (df["Vol"].isna())

deg_rate = (df.groupby(["algo","obj"])["degenerate"]
              .mean()
              .rename("degenerate_rate")
              .reset_index())

summary = (df.groupby(["algo","obj"])[metrics]
             .agg(["mean","std"])
             .reset_index())

deg_rate, summary


(        algo   obj  degenerate_rate
 0        a2c  cvar         0.333333
 1        a2c    rn         0.000000
 2  reinforce  cvar         0.666667
 3  reinforce    rn         0.000000,
         algo   obj      CAGR                 Vol              Sharpe  \
                         mean       std      mean       std      mean   
 0        a2c  cvar  0.188135  0.266621  0.069669  0.065648  2.028093   
 1        a2c    rn  1.128791  0.032836  0.149533  0.001874  5.135506   
 2  reinforce  cvar  0.219745  0.380610  0.056526  0.097906  3.073553   
 3  reinforce    rn  1.043472  0.132375  0.153941  0.010821  4.750171   
 
                   MDD               VaR_5              CVaR_5            \
         std      mean       std      mean       std      mean       std   
 0  1.576471 -0.044102  0.038195  0.003338  0.003984  0.010125  0.009680   
 1  0.164670 -0.051870  0.001015  0.011093  0.000761  0.019654  0.000172   
 2       NaN -0.025921  0.044896  0.005432  0.009409  0.007921  0.0137

In [ ]:
g = df_nd.groupby(["algo","obj"])[metrics]
summary_nd = g.agg(["mean","std","count"]).reset_index()
summary_nd


algo   obj      CAGR                       Vol                  \
                        mean       std count      mean       std count   
0        a2c  cvar  0.282202  0.298464     2  0.104504  0.036586     2   
1        a2c    rn  1.128791  0.032836     3  0.149533  0.001874     3   
2  reinforce  cvar  0.659235       NaN     1  0.169578       NaN     1   
3  reinforce    rn  1.043472  0.132375     3  0.153941  0.010821     3   

     Sharpe            ... VaR_5    CVaR_5                    EVaR_5  \
       mean       std  ... count      mean       std count      mean   
0  2.028093  1.576471  ...     2  0.015188  0.005798     2  0.032596   
1  5.135506  0.164670  ...     3  0.019654  0.000172     3  0.032252   
2  3.073553       NaN  ...     1  0.023763       NaN     1  0.035659   
3  4.750171  0.724077  ...     3  0.020230  0.002420     3  0.033159   

                   final_equity                     
        std count          mean          std count  
0  0.002008     2  13385.551078  3611.226894     2  
1  0.000136     3  24072.780754   431.740930     3  
2       NaN     1  18017.172551          NaN     1  
3  0.001821     3  22960.493239  1723.529589     3  

[4 rows x 26 columns]

In [ ]:
import glob
paths = glob.glob("/content/drive/MyDrive/rl_results/**/model_*_seed*.pt", recursive=True)
print("total model files:", len(paths))
print("\n".join(paths[:50]))  # tampilkan sebagian


#EQUITY CURVE

In [ ]:
import glob
paths = glob.glob("/content/drive/MyDrive/rl_results/**/model_*_seed*.pt", recursive=True)
print("total model files:", len(paths))
print("\n".join(paths[:50]))  # tampilkan sebagian


total model files: 12
/content/drive/MyDrive/rl_results/EVAL_20260215_100113/model_reinforce_rn_seed0.pt
/content/drive/MyDrive/rl_results/EVAL_20260215_100113/model_reinforce_rn_seed1.pt
/content/drive/MyDrive/rl_results/EVAL_20260215_100113/model_reinforce_rn_seed2.pt
/content/drive/MyDrive/rl_results/EVAL_20260215_100113/model_reinforce_cvar_seed0.pt
/content/drive/MyDrive/rl_results/EVAL_20260215_100113/model_reinforce_cvar_seed1.pt
/content/drive/MyDrive/rl_results/EVAL_20260215_100113/model_reinforce_cvar_seed2.pt
/content/drive/MyDrive/rl_results/EVAL_20260215_100113/model_a2c_rn_seed0.pt
/content/drive/MyDrive/rl_results/EVAL_20260215_100113/model_a2c_rn_seed1.pt
/content/drive/MyDrive/rl_results/EVAL_20260215_100113/model_a2c_rn_seed2.pt
/content/drive/MyDrive/rl_results/EVAL_20260215_100113/model_a2c_cvar_seed0.pt
/content/drive/MyDrive/rl_results/EVAL_20260215_100113/model_a2c_cvar_seed1.pt
/content/drive/MyDrive/rl_results/EVAL_20260215_100113/model_a2c_cvar_seed2.pt


In [ ]:
OUTDIR = "/content/drive/MyDrive/rl_results/EVAL_20260215_100113"


In [ ]:
import os
import torch
import numpy as np
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"

env_eval = MultiStockReturnEnv(returns_eval, reward_scale=1e-3)

model_files = sorted(glob.glob(os.path.join(OUTDIR, "model_*_seed*.pt")))
print("Using models:", len(model_files))

for mp in model_files:
    ckpt = torch.load(mp, map_location=device)
    algo, obj, seed = ckpt["algo"], ckpt["obj"], ckpt["seed"]

    obs_dim = env_eval.observation_space.shape[0]
    act_dim = env_eval.action_space.n

    if algo == "reinforce":
        net = PolicyBaseline(obs_dim, act_dim, hidden=256).to(device) # Changed hidden to 256
    else:
        net = ActorCriticNet(obs_dim, act_dim, hidden=256).to(device)

    net.load_state_dict(ckpt["model_state"])

    out = eval_policy(net, env_eval, device=device)

    curve_df = pd.DataFrame({
        "t": np.arange(len(out["equity"])),
        "equity": out["equity"],
        "drawdown": out["drawdown"]
    })

    curve_df.to_csv(os.path.join(OUTDIR, f"curve_{algo}_{obj}_seed{seed}.csv"), index=False)

print("Curve CSV generated.")

Using models: 12
Curve CSV generated.


In [ ]:
import matplotlib.pyplot as plt

def load_stack(pattern, col):
    files = sorted(glob.glob(pattern))
    arrs = []
    Lmin = None
    for fp in files:
        dfc = pd.read_csv(fp)
        a = dfc[col].values
        Lmin = len(a) if Lmin is None else min(Lmin, len(a))
        arrs.append(a)
    X = np.stack([a[:Lmin] for a in arrs], axis=0)
    return X

for algo in ["reinforce","a2c"]:
    for obj in ["rn","cvar"]:
        pattern = os.path.join(OUTDIR, f"curve_{algo}_{obj}_seed*.csv")
        files = glob.glob(pattern)
        if len(files) == 0:
            continue

        Xeq = load_stack(pattern, "equity")
        Xdd = load_stack(pattern, "drawdown")

        t = np.arange(Xeq.shape[1])

        # Equity
        m = Xeq.mean(axis=0)
        s = Xeq.std(axis=0, ddof=1)

        plt.figure()
        plt.plot(t, m)
        plt.fill_between(t, m-s, m+s, alpha=0.25)
        plt.title(f"Equity (Mean ± Std) | {algo}-{obj}")
        plt.xlabel("t"); plt.ylabel("equity")
        fname = os.path.join(OUTDIR, f"equity_meanstd_{algo}_{obj}.png")
        plt.savefig(fname, dpi=200, bbox_inches="tight")
        plt.close()

        # Drawdown
        m = Xdd.mean(axis=0)
        s = Xdd.std(axis=0, ddof=1)

        plt.figure()
        plt.plot(t, m)
        plt.fill_between(t, m-s, m+s, alpha=0.25)
        plt.title(f"Drawdown (Mean ± Std) | {algo}-{obj}")
        plt.xlabel("t"); plt.ylabel("drawdown")
        fname = os.path.join(OUTDIR, f"drawdown_meanstd_{algo}_{obj}.png")
        plt.savefig(fname, dpi=200, bbox_inches="tight")
        plt.close()

print("Mean ± Std figures saved.")


Mean ± Std figures saved.
